In [1]:
import time
import json
from confluent_kafka import Producer
from confluent_kafka.admin import AdminClient
import random
from datetime import datetime

ModuleNotFoundError: No module named 'confluent_kafka'

In [ ]:
#função de callback do producer. para ver se teve erro ou sucesso.

def delivery_report(err, msg):
    if err is not None:
        print('Falha : {}'.format(err))
    else:
        print('Sucesso {} [{}]'.format(msg.topic(), msg.partition()))

In [ ]:
# kafka configuration
conf = {
    'bootstrap.servers': 'kafka1:9092,kafka2:9093'
}

In [ ]:
#Remove o tópico antes de iniciar o envio
admin_client = AdminClient(conf)
admin_client.delete_topics(topics=[f"impacta"])

{'impacta': <Future at 0x7f2262057890 state=running>}

In [ ]:
# kafka producer init
producer = Producer(conf)

In [ ]:
# Função para gerar mensagens simuladas
def generate_message():
    users = ["user1", "user2", "user3"]
    actions = ["post", "like", "comment"]
    return {
        "user": random.choice(users),
        "action": random.choice(actions),
        "message": f"This is a {random.choice(actions)} by {random.choice(users)}",
        "timestamp": datetime.now().isoformat()  # Adicionar timestamp
    }

# Enviar mensagens continuamente
count = 0
flush_interval = 10
while True:
    message = generate_message()
    producer.produce(f"impacta", key=None, value=json.dumps(message), callback=delivery_report)
    count += 1
    if count % flush_interval == 0:
        producer.flush(2)
    if count % 10 == 0:
        print(f"{count} mensagens enviadas ")
    time.sleep(1)  # Esperar 1 segundo entre mensagens


Sent: {'user': 'user2', 'action': 'comment', 'message': 'This is a like by user2', 'timestamp': '2025-11-19T14:53:53.845075'}
Sent: {'user': 'user3', 'action': 'like', 'message': 'This is a like by user2', 'timestamp': '2025-11-19T14:53:54.855437'}
Sent: {'user': 'user1', 'action': 'comment', 'message': 'This is a like by user3', 'timestamp': '2025-11-19T14:53:55.858570'}
Sent: {'user': 'user1', 'action': 'like', 'message': 'This is a like by user3', 'timestamp': '2025-11-19T14:53:56.859358'}
Sent: {'user': 'user2', 'action': 'comment', 'message': 'This is a post by user3', 'timestamp': '2025-11-19T14:53:57.860140'}
Sent: {'user': 'user1', 'action': 'comment', 'message': 'This is a comment by user1', 'timestamp': '2025-11-19T14:53:58.868459'}
Sent: {'user': 'user2', 'action': 'post', 'message': 'This is a post by user1', 'timestamp': '2025-11-19T14:53:59.872517'}
Sent: {'user': 'user3', 'action': 'post', 'message': 'This is a comment by user1', 'timestamp': '2025-11-19T14:54:00.880564'

KeyboardInterrupt: 